Qual è la distribuzione dei runner utilizzati nei workflow GitHub Actions di progetti AI open-source, e in che misura la presenza di runner self-hosted o custom indica requisiti infrastrutturali particolari, come l'utilizzo di GPU per training o inference?

In [12]:
import pandas as pd

DATASET_PATH = "/Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/gigawork/dataset_with_ids.csv"
BASE_GIGAWORK_PATH = "/Users/giuseppepiosorrentino/awesome-ai-agents-cicd-extract/gigawork/all_workflows"

df = pd.read_csv(DATASET_PATH)

In [13]:
repositories = df.groupby("repository")

# per ogni repository prendiamo tutti i file hash associati allo stesso file hash, se il file è stato eliminato prendiamo l'ultimo.
tmp = df.dropna(subset=["repository", "workflow_global_id"]).copy()

# timestamp per prendere l'ultima modifica effettuata al file
tmp["event_ts"] = pd.to_numeric(tmp["committed_date"], errors="coerce")
tmp["event_ts"] = tmp["event_ts"].fillna(pd.to_numeric(tmp["authored_date"], errors="coerce"))
tmp["file_h"] = tmp["file_hash"].fillna(tmp["previous_file_hash"])

last_file = (
    tmp.sort_values(["repository", "workflow_global_id", "event_ts"])
       .groupby(["repository", "workflow_global_id"], as_index=False)
       .tail(1)
       .reset_index(drop=True)
)
repo_summary = (
    last_file.groupby("repository", as_index=False)
    .agg(
        files=("file_h", lambda s: [x for x in s.dropna().unique()]),
    )
    .reset_index(drop=True)
)

In [14]:
import yaml
from pathlib import Path
from collections import Counter
import pandas as pd

def extract_runners(job: dict) -> list[str]:
    runs_on = job.get("runs-on", "")
    strategy = job.get("strategy", {}) or {}
    
    # Aggiunto: matrix potrebbe non essere un dict
    matrix = strategy.get("matrix", {}) if isinstance(strategy, dict) else {}
    matrix = matrix if isinstance(matrix, dict) else {}
    
    os_list = matrix.get("os", [])
    os_list = os_list if isinstance(os_list, list) else []

    if isinstance(runs_on, str):
        if "matrix.os" in runs_on and os_list:
            return [str(o).strip() for o in os_list if str(o).strip()]
        return [runs_on.strip()] if runs_on.strip() else []

    if isinstance(runs_on, list):
        return [str(r).strip() for r in runs_on if str(r).strip()]

    return []

def extract_all_runners(workflow_dict: dict) -> list[str]:
    """Estrae tutti i runner da tutti i job di un workflow."""
    if not isinstance(workflow_dict, dict):
        return []

    runners = []
    for job in workflow_dict.get("jobs", {}).values():
        if isinstance(job, dict):
            runners.extend(extract_runners(job))
    return runners

stats_per_repo = {}
total_runner_counter = Counter()        # somma occorrenze totali
total_runner_counter_unique = Counter() # 1 per file (quanto è diffuso)

for repo in repo_summary["repository"].unique():
    files = repo_summary.loc[repo_summary["repository"] == repo, "files"].values[0]
    repo_runners = Counter()

    for file in files:
        path = Path(BASE_GIGAWORK_PATH) / repo / file
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = yaml.safe_load(f)
        except Exception:
            continue

        runners = extract_all_runners(data)
        if not runners:
            continue

        repo_runners.update(runners)
        total_runner_counter.update(runners)
        total_runner_counter_unique.update(set(runners))  # unico per file

    stats_per_repo[repo] = dict(repo_runners)

print("Top runner totali:", total_runner_counter.most_common(20))
print("Top runner unici per file:", total_runner_counter_unique.most_common(20))

total_files_with_runner = sum(total_runner_counter_unique.values())

runner_percentage_df = pd.DataFrame([
    {
        "runner": runner,
        "count_unique": count,
        "percentage": (count / total_files_with_runner) * 100 if total_files_with_runner else 0.0,
    }
    for runner, count in total_runner_counter_unique.items()
]).sort_values(["percentage", "runner"], ascending=[False, True]).reset_index(drop=True)



Top runner totali: [('ubuntu-latest', 836), ('macos-latest', 39), ('ubuntu-22.04', 20), ('windows-latest', 19), ('windows-2019', 15), ('blacksmith-2vcpu-ubuntu-2404', 13), ('${{ matrix.os }}', 11), ('self-hosted', 11), ('blacksmith-4vcpu-ubuntu-2204', 10), ("${{ (matrix.language == 'swift' && 'macos-latest') || 'ubuntu-latest' }}", 9), ('ubuntu-20.04', 8), ('medium', 7), ('${{ matrix.arch.runner }}', 5), ('ubuntu-24.04', 4), ('small', 4), ('${{ fromJSON(inputs.runner) }}', 4), ("${{ matrix.arch == 'arm64' && 'ubuntu-24.04-arm' || 'ubuntu-22.04' }}", 3), ('buildjet-4vcpu-ubuntu-2204', 3), ('${{ matrix.runner }}', 2), ('macos-14', 2)]
Top runner unici per file: [('ubuntu-latest', 581), ('macos-latest', 20), ('windows-latest', 18), ('ubuntu-22.04', 12), ('blacksmith-2vcpu-ubuntu-2404', 11), ('${{ matrix.os }}', 10), ("${{ (matrix.language == 'swift' && 'macos-latest') || 'ubuntu-latest' }}", 9), ('blacksmith-4vcpu-ubuntu-2204', 8), ('self-hosted', 8), ('medium', 6), ('${{ matrix.arch.runn

In [15]:
runner_percentage_df

,runner,count_unique,percentage
0,ubuntu-latest,581,80.248619
1,macos-latest,20,2.762431
2,windows-latest,18,2.486188
3,ubuntu-22.04,12,1.657459
4,blacksmith-2vcpu-ubuntu-2404,11,1.519337
5,${{ matrix.os }},10,1.381215
6,${{ (matrix.language == 'swift' && 'macos-late...,9,1.243094
7,blacksmith-4vcpu-ubuntu-2204,8,1.104972
8,self-hosted,8,1.104972
9,medium,6,0.828729


I progetti usano quasi esclusivamente ubuntu-latest (80%), il che suggerisce workflow orientati al testing leggero piuttosto che al training. I runner custom (~6-7% aggregando tutti i non-standard) appaiono in progetti che hanno esigenze di build più pesanti, ma l'ipotesi GPU non è confermata dai dati. IL self-hosted puro è solo l'1.1%. Sarebbe necessario analizzare i nomi dei job o i tool installati (es. cuda, torch) per verificare effettivamente la presenza di requisiti GPU.